# TMC-Tongue - YOLOv8 Training (full dataset, txt/YOLO format)

Dataset: **TMC-Tongue v3** (Gao Longfei, 2026) - Dryad `10.5061/dryad.1c59zw48r`, CC0 public domain.

**6,719 images / 5,594 train + 572 val + 553 test / 20 classes.**

### Why the `txt` format and not `coco`
The zip ships the same 6,719 photos three times over (`coco`, `txt`, `xml`) - identical
filenames, identical checksums, only the annotation format differs. The `txt` copy is
already in YOLO layout, so there is no conversion step to go wrong. It also has a label
file for all 5,594 training images, whereas `train.json` only covers 5,404.

### Class numbering - read this before changing anything
Three sources disagree. Verified against the actual boxes:

| source | 17 | 18 | 19 | 20 |
|---|---|---|---|---|
| in-zip `shezhenv3-txt.yaml` | piweiao | xinfeitu | xinfeiao | - |
| Dryad abstract + GitHub README prose | piweiao | xinfeitu | xinfeiao | - |
| GitHub `YOLO verification code` yaml | piweiao | **xinfeiao** | **xinfeitu** | - |
| `coco/train.json` | piweiao | piweitu (0 boxes) | xinfeiao | xinfeitu |

The GitHub yaml is the outlier and is wrong. Class 19 carries 325 boxes in the txt labels
and COCO calls that same set `xinfeiao` (320 boxes); COCO's `xinfeitu` has 2. Following the
GitHub yaml would flip heart/lung depression and protrusion - opposite TCM readings - on
every detection. **This notebook uses the in-zip ordering.**

`piweitu` has zero annotations anywhere, which is why the txt format drops it: 21 -> 20 classes.

### Steps
1. Runtime -> Change runtime type -> **T4 GPU**
2. Run all cells top to bottom
3. Collect `tongue_yolo_best.pt` and `class_reliability.json` from Drive

You do not need to upload anything. Cell 3 looks in Drive for a zip or a **complete**
extracted copy; if it finds neither (or only a partial one), Cell 4 pulls the zip from
Dryad directly and caches it to Drive so it happens only once.


In [ ]:
# Cell 1 - GPU check
!nvidia-smi
import torch
print(f"\nPyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
# Cell 2 - install
!pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

In [ ]:
# Cell 3 - mount Drive and work out what we actually have
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
EXPECTED = {'train': 5594, 'val': 572, 'test': 553}

SOURCE_ZIP = None
SOURCE_DIR = None

# 1) any shezhen zip anywhere in Drive
for p in ROOT.rglob('*.zip'):
    if 'shezhen' in p.name.lower():
        SOURCE_ZIP = p
        print(f"Found zip: {SOURCE_ZIP}  ({SOURCE_ZIP.stat().st_size/1e9:.2f} GB)")
        break

# 2) otherwise look for a COMPLETE txt-format tree already sitting in Drive
if SOURCE_ZIP is None:
    print("No zip in Drive. Inspecting any shezhen folders ...\n")
    for cand in ROOT.rglob('*'):
        if not cand.is_dir():
            continue
        if not all((cand / s / 'images').is_dir() for s in EXPECTED):
            continue
        counts = {
            s: sum(1 for f in (cand / s / 'images').iterdir() if f.suffix.lower() in IMG_EXT)
            for s in EXPECTED
        }
        has_labels = all((cand / s / 'labels').is_dir() for s in EXPECTED)
        complete = has_labels and all(counts[s] == EXPECTED[s] for s in EXPECTED)
        status = "COMPLETE" if complete else "incomplete"
        print(f"  {cand}")
        print(f"    {counts}  labels={has_labels}  -> {status}")
        if complete:
            SOURCE_DIR = cand
            break

if SOURCE_ZIP is None and SOURCE_DIR is None:
    print("\nNothing usable in Drive.")
    print("The next cell will download the dataset straight from Dryad instead")
    print("(public domain, ~2.34 GB, a few minutes on Colab's connection).")
else:
    print(f"\nUsing: {SOURCE_ZIP or SOURCE_DIR}")

In [ ]:
# Cell 4 - stage to local disk (only the txt copy: ~1/3 the files)
import zipfile, shutil, time
from pathlib import Path

LOCAL = Path('/content/tmc')
if LOCAL.exists():
    shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)

t0 = time.time()
staged = Path('/content/source.zip')

if SOURCE_ZIP is None and SOURCE_DIR is None:
    # Straight from the source. CC0, so this is entirely above board, and Colab's
    # connection is far quicker than uploading 2.34 GB from a laptop.
    DRYAD_URL = 'https://datadryad.org/downloads/file_stream/4540656'
    print("Downloading TMC-Tongue from Dryad ...")
    !wget -q --show-progress -O {staged} {DRYAD_URL}

    if not staged.exists() or staged.stat().st_size < 1e9:
        raise SystemExit(
            f"Download failed or truncated ({staged.stat().st_size/1e6:.0f} MB). "
            "Fetch shezhen_datasets1.zip manually from "
            "https://datadryad.org/dataset/doi:10.5061/dryad.1c59zw48r "
            "and drop it in your Drive, then re-run from Cell 3."
        )
    print(f"  {staged.stat().st_size/1e9:.2f} GB in {time.time()-t0:.0f}s")

    # Keep a copy so this only ever happens once
    try:
        shutil.copy2(staged, ROOT / 'shezhen_datasets1.zip')
        print("  Saved to Drive for next time")
    except Exception as e:
        print(f"  (couldn't cache to Drive: {e})")

    SOURCE_ZIP = staged

if SOURCE_ZIP:
    if SOURCE_ZIP != staged:
        # Copy off Drive first - unzipping over the FUSE mount is glacially slow
        print("Copying zip off Drive ...")
        shutil.copy2(SOURCE_ZIP, staged)
        print(f"  {time.time()-t0:.0f}s")

    with zipfile.ZipFile(staged) as zf:
        members = [m for m in zf.namelist() if 'shezhenv3-txt' in m and not m.endswith('/')]
        print(f"Extracting {len(members)} txt-format files ...")
        for i, m in enumerate(members):
            zf.extract(m, LOCAL)
            if i % 2000 == 0:
                print(f"  {i}/{len(members)}", end='\r')
    print(f"\n  {time.time()-t0:.0f}s total")
else:
    print("Copying folder off Drive ...")
    shutil.copytree(SOURCE_DIR, LOCAL / 'shezhenv3-txt')
    print(f"  {time.time()-t0:.0f}s")

# Find the folder that actually holds train/val/test
DATA_ROOT = None
for p in LOCAL.rglob('*'):
    if p.is_dir() and all((p / s / 'images').is_dir() for s in ('train', 'val', 'test')):
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    raise SystemExit("Could not find train/val/test under the extracted files.")
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
# Cell 5 - integrity check. Fails loudly rather than quietly training on a fraction.
from collections import Counter
from pathlib import Path

EXPECTED = {'train': 5594, 'val': 572, 'test': 553}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

train_counts = Counter()
problems = []

for split, expected in EXPECTED.items():
    imgs = [p for p in (DATA_ROOT / split / 'images').iterdir() if p.suffix.lower() in IMG_EXT]
    lbls = list((DATA_ROOT / split / 'labels').glob('*.txt'))

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}
    orphans   = img_stems ^ lbl_stems

    print(f"{split:<6} images {len(imgs):>5} (expected {expected})   labels {len(lbls):>5}   unmatched {len(orphans)}")

    if len(imgs) != expected:
        problems.append(f"{split}: {len(imgs)} images, expected {expected}")
    if orphans:
        problems.append(f"{split}: {len(orphans)} images/labels without a partner")

    if split == 'train':
        for p in lbls:
            for line in p.read_text().splitlines():
                if line.strip():
                    train_counts[int(line.split()[0])] += 1

if problems:
    raise SystemExit("Dataset is incomplete:\n  " + "\n  ".join(problems))

print("\nAll splits complete.\n")

CLASS_NAMES = [
    'jiankangshe', 'botaishe', 'hongshe', 'zishe', 'pangdashe',
    'shoushe', 'hongdianshe', 'liewenshe', 'chihenshe', 'baitaishe',
    'huangtaishe', 'heitaishe', 'huataishe', 'shenquao', 'shenqutu',
    'gandanao', 'gandantu', 'piweiao', 'xinfeitu', 'xinfeiao',
]

max_id = max(train_counts)
if max_id > 19:
    raise SystemExit(f"Found class id {max_id}; expected 0-19. Wrong annotation format?")

MIN_INSTANCES = 50
UNRELIABLE = []

print(f"{'id':>3}  {'class':<13} {'boxes':>6}")
for i, name in enumerate(CLASS_NAMES):
    n = train_counts.get(i, 0)
    flag = ''
    if n < MIN_INSTANCES:
        flag = '  <-- too few to learn'
        UNRELIABLE.append(i)
    print(f"{i:>3}  {name:<13} {n:>6}{flag}")

print(f"\nTotal boxes: {sum(train_counts.values())}")
print(f"Suppress at inference: {[CLASS_NAMES[i] for i in UNRELIABLE]}")

In [ ]:
# Cell 6 - dataset.yaml
from pathlib import Path

YAML_PATH = Path('/content/tmc_tongue.yaml')
YAML_PATH.write_text(
    "# TMC-Tongue v3 - doi:10.5061/dryad.1c59zw48r\n"
    "# Class order follows the in-zip shezhenv3-txt.yaml (verified against COCO boxes).\n"
    f"path: {DATA_ROOT}\n"
    "train: train/images\n"
    "val: val/images\n"
    "test: test/images\n\n"
    "nc: 20\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES))
)
print(YAML_PATH.read_text())

In [ ]:
# Cell 7 - train
from ultralytics import YOLO

MODEL   = 'yolov8s.pt'   # 'yolov8n.pt' if you want it faster, 'yolov8m.pt' needs Colab Pro
EPOCHS  = 100
BATCH   = 32             # drop to 16 if you hit CUDA OOM
IMGSZ   = 640

model = YOLO(MODEL)

results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    project='/content/runs',
    name='tmc_tongue',
    exist_ok=True,
    patience=25,

    # Colour is diagnostic here - a red tongue must not be augmented into a pale one.
    hsv_h=0.010,
    hsv_s=0.30,
    hsv_v=0.25,

    # Geometry is safe: tongues get photographed at odd angles anyway.
    degrees=10,
    translate=0.10,
    scale=0.30,
    fliplr=0.5,
    flipud=0.0,      # a tongue is never upside down
    mosaic=1.0,
    close_mosaic=10,

    optimizer='auto',
    cos_lr=True,
    warmup_epochs=3,
    seed=0,
    save_period=10,  # checkpoint every 10 epochs in case the session dies
)
print("Best weights:", results.save_dir)

In [ ]:
# Cell 8 - evaluate on the held-out test split, per class
from ultralytics import YOLO
from pathlib import Path

best = Path('/content/runs/tmc_tongue/weights/best.pt')
m = YOLO(str(best))

metrics = m.val(data=str(YAML_PATH), split='test', imgsz=IMGSZ, device=0)

print(f"\nOverall   mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}   "
      f"P {metrics.box.mp:.3f}   R {metrics.box.mr:.3f}\n")

print(f"{'class':<13} {'train boxes':>11} {'mAP50':>7} {'P':>7} {'R':>7}")
for idx, c in enumerate(metrics.box.ap_class_index):
    print(f"{CLASS_NAMES[c]:<13} {train_counts.get(int(c), 0):>11} "
          f"{metrics.box.ap50[idx]:>7.3f} {metrics.box.p[idx]:>7.3f} {metrics.box.r[idx]:>7.3f}")

missing = sorted(set(range(20)) - set(int(c) for c in metrics.box.ap_class_index))
if missing:
    print("\nAbsent from the test set entirely:", [CLASS_NAMES[i] for i in missing])

In [ ]:
# Cell 9 - export weights + a reliability map the app can enforce
import json, shutil
from pathlib import Path

out = Path('/content/drive/MyDrive/tcm_platform_models')
out.mkdir(parents=True, exist_ok=True)

shutil.copy2('/content/runs/tmc_tongue/weights/best.pt', out / 'tongue_yolo_best.pt')

per_class = {}
for idx, c in enumerate(metrics.box.ap_class_index):
    per_class[CLASS_NAMES[int(c)]] = round(float(metrics.box.ap50[idx]), 4)

reliability = {
    'dataset': 'TMC-Tongue v3 (doi:10.5061/dryad.1c59zw48r)',
    'class_order_source': 'in-zip shezhenv3-txt.yaml, verified against coco/train.json boxes',
    'nc': 20,
    'names': CLASS_NAMES,
    'train_instances': {CLASS_NAMES[i]: train_counts.get(i, 0) for i in range(20)},
    'train_images': 5594,
    'val_images': 572,
    'test_images': 553,
    'min_instances_threshold': MIN_INSTANCES,
    'suppress_at_inference': [CLASS_NAMES[i] for i in UNRELIABLE],
    'test_map50_per_class': per_class,
    'test_map50': round(float(metrics.box.map50), 4),
    'test_map50_95': round(float(metrics.box.map), 4),
}
(out / 'class_reliability.json').write_text(json.dumps(reliability, indent=2))

shutil.copy2('/content/runs/tmc_tongue/results.csv', out / 'results.csv')
for png in Path('/content/runs/tmc_tongue').glob('*.png'):
    shutil.copy2(png, out / png.name)

print("Written to", out)
for f in sorted(out.iterdir()):
    print('  ', f.name)

from google.colab import files
files.download('/content/runs/tmc_tongue/weights/best.pt')
files.download(str(out / 'class_reliability.json'))

## After training

Copy both files into `tcm_platform\models\`:

- `tongue_yolo_best.pt`
- `class_reliability.json`

Then in the app:

1. `tongue_detector.py` - load `class_reliability.json`, use its `names` list (20 entries,
   **not** the old 21-entry one), and drop any detection whose class is in
   `suppress_at_inference`.
2. `app.py:166` - replace the hardcoded `"Model ready - 19,585 tongue images"` with the real
   figure from `class_reliability.json`. 19,585 came from counting the same 6,719 photos once
   per annotation format.

### Reading the results honestly

Expect a respectable headline mAP50 driven mostly by `baitaishe`, `hongdianshe`, `liewenshe`
and `chihenshe`, which between them hold 9,766 of the 15,219 training boxes. The rare classes
will score near zero no matter how long you train - `shenqutu`, `gandantu` and `xinfeitu`
have a single example each, and `jiankangshe` has 11.

`jiankangshe` is worth dwelling on: with 11 training boxes the model has effectively never
seen a healthy tongue, so it cannot conclude "healthy". Anything the app says about normality
is coming from Claude Vision and the vitals, not from YOLO.
